In [ ]:
# how to import all the tools

from src.tools.registry import TOOLS, auto_import_tools, export_tools_doc
auto_import_tools()
print(TOOLS)
print("已注册工具：", list(TOOLS.keys()))
print(export_tools_doc("markdown"))


In [2]:
import sys
sys.path.append("/home/lorraine/jingwei_QA")

In [1]:
import re
import os
import xarray as xr

def load_regular_data(zarr_path: str, time_range: tuple[str, str] = None, depth_range: tuple[float, float] = None, 
                      lat_range: tuple[float, float] = None, lon_range: tuple[float, float] = None) -> xr.DataArray:
    fname = os.path.basename(zarr_path)
    match = re.match(r"recon_([a-zA-Z0-9]+)_", fname)
    if not match:
        raise ValueError(f"Cannot parse variable name from file: {fname}")
    var_name = match.group(1)

    ds = xr.open_zarr(zarr_path)
    if var_name not in ds:
        raise KeyError(f"Variable '{var_name}' not found in {fname}. "
                       f"Available: {list(ds.data_vars)}")
    da = ds[var_name]

    if time_range is not None:
        da = da.sel(time=slice(time_range[0], time_range[1]))
    if depth_range is not None:
        da = da.sel(depth=slice(depth_range[0], depth_range[1]))
    if lat_range is not None:
        da = da.sel(lat=slice(lat_range[0], lat_range[1]))
    if lon_range is not None:
        da = da.sel(lon=slice(lon_range[0], lon_range[1]))

    return da

path = "/data1/lorraine-20419/jingwei_QA/reconstruction/recon_o2_sjtu_jingwei_v1_1.zarr"
da = load_regular_data(path, time_range=("2000-01-01", "2010-12-31"), depth_range=(0, 100), lat_range=(20, 30), lon_range=(120, 130))
print(da)


<xarray.DataArray 'o2' (time: 132, depth: 21, lat: 10, lon: 10)> Size: 2MB
dask.array<getitem, shape=(132, 21, 10, 10), dtype=float64, chunksize=(12, 21, 10, 10), chunktype=numpy.ndarray>
Coordinates:
  * depth    (depth) float32 84B 0.0 5.0 10.0 15.0 20.0 ... 85.0 90.0 95.0 100.0
  * lat      (lat) float32 40B 20.5 21.5 22.5 23.5 24.5 25.5 26.5 27.5 28.5 29.5
  * lon      (lon) float32 40B 120.5 121.5 122.5 123.5 ... 127.5 128.5 129.5
  * time     (time) datetime64[ns] 1kB 2000-01-01 2000-02-01 ... 2010-12-01
Attributes:
    long_name:  Dissolved Oxygen
    units:      μmol·kg^-1


In [4]:
import xarray as xr
import numpy as np
import pandas as pd
from typing import Optional, Dict, Any
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def compute_basic_stat(data: xr.DataArray, op: str, dim: str | list[str] | None = None) -> xr.DataArray:
    op = op.lower()
    if op == "count":
        result = data.count(dim=dim)
    elif op == "max":
        result = data.max(dim=dim)
    elif op == "min":
        result = data.min(dim=dim)
    elif op == "mean":
        result = data.mean(dim=dim)
    elif op == "median":
        result = data.median(dim=dim)
    elif op == "sum":
        result = data.sum(dim=dim)
    else:
        raise ValueError(f"Unsupported operation '{op}'. "
                         f"Choose from ['count','max','min','mean','median','sum'].")

    return result

da_2 = compute_basic_stat(da, op="mean", dim=["depth", "lat", "lon"])
print(da_2)

<xarray.DataArray 'o2' (time: 132)> Size: 1kB
dask.array<mean_agg-aggregate, shape=(132,), dtype=float64, chunksize=(12,), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 1kB 2000-01-01 2000-02-01 ... 2010-12-01


In [6]:
def fit_1D_trend(data: xr.DataArray, method: str = None) -> Dict[str, Any]:
    if data.ndim != 1:
        raise ValueError("Input DataArray must be 1D.")

    x = np.arange(len(data))
    y = data.values.astype(float)

    def linear(x, a, b):
        return a * x + b
    def quadratic(x, a, b, c):
        return a * x**2 + b * x + c
    def exponential(x, a, b):
        return a * np.exp(b * x)
    def sinusoidal(x, a, b, c, d):
        return a * np.sin(b * x + c) + d
    
    models = {
        "linear": (linear, [0, np.mean(y)]),
        "quadratic": (quadratic, [0, 0, np.mean(y)]),
        "exponential": (exponential, [1, 0.01]),
        "sinusoidal": (sinusoidal, [np.std(y), 2*np.pi/12, 0, np.mean(y)]),
    }

    results = {}

    for name, (func, p0) in models.items():
        try:
            params, _ = curve_fit(func, x, y, p0=p0, maxfev=10000)
            y_fit = func(x, *params)
            metrics = {
                "rmse": float(np.sqrt(mean_squared_error(y, y_fit))),
                "mae": float(mean_absolute_error(y, y_fit)),
                "r2": float(r2_score(y, y_fit)),
            }
            results[name] = {"params": params, "fitted": y_fit, "metrics": metrics}
        except Exception:
            continue
    
    if method is None:
        method = max(results.keys(), key=lambda m: results[m]["metrics"]["r2"])
    if method not in results:
        raise ValueError(f"Method '{method}' failed. Available: {list(results.keys())}")
    chosen = results[method]

    fitted_data = xr.DataArray(
        chosen["fitted"],
        coords=data.coords,
        dims=data.dims,
        attrs={**data.attrs, "fitted_by": method}
    )

    param_str = ", ".join([f"{p:.4f}" for p in chosen["params"]])
    equation = f"{method} fit with params: [{param_str}]"

    return {
        "fitted_data": fitted_data,
        "equation": equation,
        "metrics": chosen["metrics"],
        "method": method,
    }

fit_res = fit_1D_trend(da_2)
print(fit_res["equation"])
print(fit_res["metrics"])

sinusoidal fit with params: [6.9432, 0.5241, 0.6721, 205.8831]
{'rmse': 0.9710681097188295, 'mae': 0.786751158484715, 'r2': 0.9623305863193826}


In [3]:
from src.tools.data_loader import load_data_with_geolocator

path = "/data1/lorraine-20419/jingwei_QA/reconstruction/recon_o2_sjtu_jingwei_v1_1.zarr"
data = load_data_with_geolocator(path, time_range=("2000-01-01", "2010-12-31"), depth_range=(0, 100), 
                                geofeature="water_body", loc_name="Arctic Ocean")
print(data)

DataSourceError: assets/NaturalEarth/10m_cultural/ne_10m_admin_0_countries.shp: No such file or directory